In [1]:
import glob
import gc
import csv
import math
import os
import random
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import random
from collections import deque

from sentence_transformers import SentenceTransformer

### Encoder for damage text
encoder = SentenceTransformer("all-MiniLM-L6-v2")

sys.path.append("/kaggle/input/datasets/kiyotah/cg-lib")

from cg.api import (
    AreaType,
    Card,
    Observation,
    OptionType,
    Option,
    PlayerState,
    Pokemon,
    SearchState,
    SelectContext,
    all_attack,
    all_card_data,
    search_begin,
    search_end,
    search_step,
    to_observation_class,
)
from cg.game import battle_start, battle_finish, battle_select, visualize_data

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (3 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [2]:
all_card = all_card_data()
# Create a lookup table (dictionary) to quickly access card data by its cardId
card_table = {c.cardId:c for c in all_card}

## Deck Loading and Generation

We will load pre-defined decks from CSV files and then modify the `generate_random_deck` function to randomly select one of these loaded decks for each battle.

In [3]:
import pandas as pd
import glob
import random
import os

# Path to the directory containing deck CSV files
deck_files_path = '/kaggle/input/datasets/yu0307/deck-csv'
predefined_decks = []
predefined_deck_names = []

def load_predefined_decks(directory_path):
    loaded_decks = []
    loaded_names = []
    csv_files = sorted(glob.glob(os.path.join(directory_path, '*.csv')))

    if not csv_files:
        print(f"Warning: No CSV files found in {directory_path}. Cannot load predefined decks.")
        return [], []

    for file_path in csv_files:
        try:
            df = pd.read_csv(file_path, header=None)
            if len(df.columns) == 0:
                print(f"Warning: CSV file {file_path} is empty. Skipping.")
                continue

            deck_cards = df.iloc[:, 0].tolist()
            assert len(deck_cards) == 60, (
                f"Deck {file_path} has {len(deck_cards)} cards, but expected 60."
            )

            loaded_decks.append(deck_cards)
            loaded_names.append(os.path.splitext(os.path.basename(file_path))[0])
        except Exception as e:
            print(f"Error loading deck from {file_path}: {e}")

    print(f"Loaded {len(loaded_decks)} predefined decks from {directory_path}, each with 60 cards.")
    return loaded_decks, loaded_names

predefined_decks, predefined_deck_names = load_predefined_decks(deck_files_path)
print("Decks loaded. Training and evaluation will use randomly selected decks from this pool.")


Loaded 16 predefined decks from /kaggle/input/datasets/yu0307/deck-csv, each with 60 cards.
Decks loaded. Training and evaluation will use randomly selected decks from this pool.


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# Encoding for the game state

In [5]:
def encode_text(text):
    ### For now don't encode text, will look at encoding the "results" of the attack into an actual encoding, can do this quickly once I have a working agent.
    return torch.zeros(1)

In [6]:
def encode_energy(cost):
    vec = torch.zeros(12)
    for energy in cost:
        vec[energy] += 1
    return vec

In [7]:
ATTACK_SIZE = 14
def encode_damage(damage):
    return torch.tensor([int(damage) / 300.0])

def encode_attack(attack):
    energy = encode_energy(attack.energies)
    damage = encode_damage(attack.damage)
    text = encode_text(attack.text)
    return torch.cat([energy,damage,text])

In [8]:
MAX_ATTACKS = 4
MAX_RETREAT = 4
def encode_attacks(attacks):
    ### Randomize where the attacks are in the encoding, otherwise it will learn the attack in later sections less.
    if not attacks:
        return torch.zeros((MAX_ATTACKS, 14))
    encoded = []
    x = 0
    random_integers = [random.randint(0, MAX_ATTACKS - 1) for _ in range(MAX_ATTACKS)]
    for r in random_integers:
        try:
            enc = encode_attack(attacks[r])
        except:
            enc = torch.zeros(ATTACK_SIZE)
        encoded.append(enc)
        encoded = encoded[:MAX_ATTACKS]
    return torch.stack(encoded)

    ### Maybe try to encode stage as a percent of its total stage?
def stage(card_data):
    stage = 0
    if card_data.stage1:
        stage = 1
    if card_data.stage2:
        stage = 2
    return stage

def encode_id(card_data):
    id = card.cardId
    if id>1267:
        print(id)
        id = 1267
    return torch.tensor([id/1267])

def encode_energyType(energy):
    return torch.tensor([int(energy)/12.0])

    ### type encoding using one hot, maybe change but see.
def encode_type(card_data):
    vec = torch.zeros(14)
    vec[card_data.cardType] = 1
    vec[7] = card_data.ex
    vec[8] = card_data.megaEx
    vec[9] = card_data.tera
    vec[10] = card_data.aceSpec
    vec[11+stage(card_data)] = 1
    return vec

def encode_resistance(type):
    if type == None:
        type = 12
    return torch.tensor([int(type)/12.0])

def encode_retreat(retreat):
    return torch.tensor([int(retreat)/4])

    ###fix this later, only works for one ability
def encode_abilities(abilities):
    if not abilities:
        return torch.zeros((1,384))
    else:
        text = abilities[0].text
    embedding = encoder.encode(text)
    return torch.tensor(embedding)

def encode_hp(card):
    if not card.hp:
        return torch.zeros(1)
    return torch.tensor([float(card.hp)/380])

In [9]:
POKEMON_SIZE = 472
CARD_SIZE = 459
ZERO_CARD = torch.zeros(CARD_SIZE)
ZERO_POKEMON = torch.zeros(POKEMON_SIZE)

In [10]:
CARD_ENCODING_CACHE = {}
import random
def encode_card(card_id):
    if card_id in CARD_ENCODING_CACHE:
        return CARD_ENCODING_CACHE[card_id]

    card = card_table[card_id]

    typed = encode_type(card)
    energy = encode_energyType(card.energyType)
    attacks = torch.flatten(encode_attacks(card.attacks))
    resistance = encode_resistance(card.resistance)
    weakness = encode_resistance(card.weakness)
    retreat = encode_retreat(card.retreatCost)
    abilities = torch.flatten(encode_abilities(card.skills))
    hp = encode_hp(card)

    encoded = torch.cat([
        hp,
        attacks,
        abilities,
        energy,
        typed,
        resistance,
        weakness,
        retreat
    ])

    CARD_ENCODING_CACHE[card_id] = encoded

    return encoded


In [11]:
def encode_pokemon(pokemon):
    appear = torch.tensor(pokemon.appearThisTurn)
    energy = encode_energy(pokemon.energies)
    ###quick search for max hp online
    hp = torch.tensor([int(pokemon.hp)/1500])
    card_encoding = encode_card(pokemon.id)
    encoded = torch.cat([hp,energy,card_encoding])
    return encoded

In [12]:
def encode_stadium(obs):
    if len(obs.current.stadium) == 0:
        return ZERO_CARD
    return encode_card(obs.current.stadium[0].id)


def encode_active(active):
    if len(active) == 0 or active[0] is None:
        return ZERO_POKEMON
    return encode_pokemon(active[0])


def encode_bench(bench):
    pokemons = []

    for pokemon in bench[:8]:
        pokemons.append(encode_pokemon(pokemon))

    while len(pokemons) < 8:
        pokemons.append(ZERO_POKEMON)

    return torch.stack(pokemons)


def encode_player_state(state):
    active = encode_active(state.active)
    bench = encode_bench(state.bench)

    vec = torch.tensor([
        state.asleep,
        state.burned,
        state.confused,
        state.paralyzed,
        state.poisoned
    ], dtype=torch.float32)

    deck_count = torch.tensor([
        int(state.deckCount) / 60
    ], dtype=torch.float32)

    return torch.cat([
        active,
        bench.flatten(),
        vec,
        deck_count
    ])


def encode_active_state(state):
    hand = encode_hand(state.hand)
    rest = encode_player_state(state)

    return torch.cat([
        hand.flatten(),
        rest
    ])


def encode_hand(hand):
    cards = []

    for card in hand[:10]:
        cards.append(encode_card(card.id))

    while len(cards) < 10:
        cards.append(ZERO_CARD)

    return torch.stack(cards)

In [13]:
import time
def encode_state(obs):
    ###could be more efficient by encoding just what is useful for stadium card (skill maybe)
    stadium = encode_stadium(obs)
    index = obs.current.yourIndex
    opp = 0
    if index == 1:
        opp = 0
    else:
        opp = 1
    player = encode_active_state(obs.current.players[index])
    opponent = encode_player_state(obs.current.players[opp])
    encoded = torch.cat([stadium,player,opponent])
    return encoded

## Neural Network Value Function paired with Tree Search



Value Network

In [14]:
class StateValueNetwork(nn.Module):
    def __init__(self, input_dim):
        super(StateValueNetwork, self).__init__()
        # Dense layers to process the tabular state vector
        self.fc1 = nn.Linear(input_dim, 256)
        self.fc2 = nn.Linear(256, 256)
        self.fc3 = nn.Linear(256, 128)
        
        # Value head outputting a single scalar score
        self.value_head = nn.Linear(128, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        # Tanh maps output cleanly between -1 (Loss) and +1 (Win)
        return torch.tanh(self.value_head(x))

Self-Play & Search Loop

In [15]:
class Node:
    max_option: [int]
    state: SearchState
    max_end: float
    obs: Observation
    def __init__(self, state: SearchState):
        self.max_end = -1.0
        self.state = state
        self.max_option = None

    def option_value(self, option:[int], max_end:float,):
        if max_end > self.max_end:
            self.max_option = option
            self.max_end = max_end

# Optimal path DFS

In [16]:
from itertools import combinations

# Search limits are important here. The previous version performed a full
# recursive traversal of every combination, which can become enormous.
SEARCH_MAX_DEPTH = 8
SEARCH_MAX_NODES = 250
MAX_BRANCHES = 48


def evaluate_state_value(network, obs):
    """Evaluate a state without creating autograd graphs."""
    with torch.inference_mode():
        return network(encode_state(obs).to(device)).item()


def _selection_key(selection):
    return tuple(sorted(selection))


def _search_key(cur, depth):
    # searchId identifies the current search state. Include the selected
    # options and depth so repeated search states cannot recurse forever.
    return (getattr(cur.state, "searchId", None), depth)


def optimal_path(cur, network, depth=0, budget=None, seen=None):
    if budget is None:
        budget = [SEARCH_MAX_NODES]
    if seen is None:
        seen = set()

    # Hard stop: never let one game turn consume unbounded search time.
    if depth >= SEARCH_MAX_DEPTH or budget[0] <= 0:
        value = evaluate_state_value(network, cur.state.observation)
        cur.max_end = value
        cur.max_option = None
        return cur

    key = _search_key(cur, depth)
    if key in seen:
        value = evaluate_state_value(network, cur.state.observation)
        cur.max_end = value
        cur.max_option = None
        return cur
    seen.add(key)
    budget[0] -= 1

    select = cur.state.observation.select
    n = len(select.option)

    if n == 0:
        cur.max_end = evaluate_state_value(network, cur.state.observation)
        return cur

    # Do not search every possible cardinality. In particular, an empty
    # selection can return the same search state and cause infinite recursion.
    # Prefer the normal maxCount action; fall back to minCount only if needed.
    counts = [select.maxCount]
    if select.minCount != select.maxCount:
        counts.append(select.minCount)

    selections = []
    for count in counts:
        if count < 0 or count > n:
            continue
        if count == 0:
            # Avoid the common no-op -> same-state recursion.
            continue
        selections.extend(combinations(range(n), count))

    if not selections:
        cur.max_end = evaluate_state_value(network, cur.state.observation)
        return cur

    # Bound branching. Keep terminal (type 14) selections first because they
    # correspond to actual game actions and are what dfs_agent ultimately needs.
    terminal = [s for s in selections if all(select.option[i].type == 14 for i in s)]
    non_terminal = [s for s in selections if s not in terminal]
    selections = terminal + non_terminal
    if len(selections) > MAX_BRANCHES:
        selections = selections[:MAX_BRANCHES]

    # This is a value function, so we are choosing the highest-valued branch.
    for selection in selections:
        if budget[0] <= 0:
            break

        selection = list(selection)

        # A complete game action: record it as the candidate action.
        if all(select.option[i].type == 14 for i in selection):
            value = evaluate_state_value(network, cur.state.observation)
        else:
            try:
                child_state = search_step(
                    cur.state.searchId,
                    selection
                )
                child = Node(child_state)
                child = optimal_path(
                    child, network, depth + 1, budget=budget, seen=seen
                )
                value = child.max_end
            except Exception:
                # A malformed search branch should not kill the entire move.
                continue

        cur.option_value(selection, value)

    # If every branch failed, evaluate the current state rather than returning
    # the default -1 forever.
    if cur.max_option is None:
        cur.max_end = evaluate_state_value(network, cur.state.observation)

    return cur


# Agent

Random agent

In [17]:
def random_agent(obs_dict: dict) -> list[int]:
    return random.sample(list(range(len(obs_dict["select"]["option"]))), obs_dict["select"]["maxCount"])

DFS_agent

In [18]:
import re
class TrainingState:
    """Deck-specific exploration/performance state using one shared value network."""
    network: StateValueNetwork
    epsilon: float
    epsilon_min: float
    epsilon_decay: float
    deck: list[int]
    deck_name: str

    def __init__(
        self,
        network: StateValueNetwork,
        deck: list[int],
        optimizer,
        deck_name="unnamed_deck",
        epsilon=1.0,
        epsilon_min=0.1,
        epsilon_decay=0.995,
    ):
        self.network = network
        self.deck = deck
        self.deck_name = deck_name
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.optimizer = optimizer
        self.loss_fn = nn.MSELoss()

        # Low-epsilon performance tally. These only count games played after
        # this deck reaches the evaluation threshold.
        self.performance = {
            "games": 0,
            "wins": 0,
            "losses": 0,
            "draws": 0,
        }
        self.evaluation_started = False

    def update_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

    def train(self, states, targets, max_batch_size=256):
        """Train on a bounded batch so CPU RAM cannot grow with generation size."""
        if not states:
            return None
        if len(states) != len(targets):
            raise ValueError(f"States: {len(states)}, Targets: {len(targets)}")

        # Hard cap: torch.stack can otherwise create a very large temporary
        # tensor on top of the original list of individual tensors.
        if len(states) > max_batch_size:
            indices = random.sample(range(len(states)), max_batch_size)
            states = [states[i] for i in indices]
            targets = [targets[i] for i in indices]

        batch_states = torch.stack(states).to(device)
        batch_targets = torch.as_tensor(
            targets, dtype=torch.float32, device=device
        ).reshape(-1, 1)

        self.network.train()
        self.optimizer.zero_grad(set_to_none=True)
        predictions = self.network(batch_states)
        loss = self.loss_fn(predictions, batch_targets)
        loss.backward()
        self.optimizer.step()

        loss_value = float(loss.detach().cpu())

        # Explicitly release large temporaries before returning to the game loop.
        del batch_states, batch_targets, predictions, loss
        return loss_value

    def record_result(self, player_index, winner_index, threshold=0.10):
        """Count only post-exploration games once this deck is sufficiently trained."""
        if not self.evaluation_started and self.epsilon <= threshold:
            self.evaluation_started = True
            print(f"Evaluation started for deck {self.deck_name} (epsilon={self.epsilon:.3f})")

        if not self.evaluation_started:
            return

        self.performance["games"] += 1
        if winner_index < 0:
            self.performance["draws"] += 1
        elif winner_index == player_index:
            self.performance["wins"] += 1
        else:
            self.performance["losses"] += 1

    def win_rate(self):
        games = self.performance["games"]
        return self.performance["wins"] / games if games else 0.0

    def save_deck_metadata(self, directory="trainingState"):
        os.makedirs(directory, exist_ok=True)
        safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", self.deck_name)
        deck_path = os.path.join(directory, f"deck_{safe_name}.csv")
        stats_path = os.path.join(directory, f"performance_{safe_name}.csv")
        with open(deck_path, "w", newline="") as f:
            csv.writer(f).writerow(self.deck)
        with open(stats_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["games", "wins", "losses", "draws", "win_rate", "epsilon"])
            writer.writerow([
                self.performance["games"],
                self.performance["wins"],
                self.performance["losses"],
                self.performance["draws"],
                self.win_rate(),
                self.epsilon,
            ])


def deck_label(deck):
    """Stable short deck label for console output/files."""
    return "_".join(map(str, deck[:6]))


print("TrainingState updated: bounded batches, shared optimizer, epsilon tracking, and performance tallies.")


TrainingState updated: bounded batches, shared optimizer, epsilon tracking, and performance tallies.


In [19]:
def dfs_agent(obs_dict: dict, tState: TrainingState):
    your_deck = tState.deck
    network = tState.network
    epsilon = tState.epsilon

    if random.random() < epsilon:
        return random_agent(obs_dict)

    obs = to_observation_class(obs_dict)

    your_index = obs.current.yourIndex
    state = obs.current
    active = state.players[1 - your_index].active

    search_state = search_begin(
        obs,
        your_deck=random.sample(
            your_deck,
            state.players[your_index].deckCount
        ),
        your_prize=random.sample(
            your_deck,
            len(state.players[your_index].prize)
        ),
        opponent_deck=[1072] * state.players[1 - your_index].deckCount,
        opponent_prize=[1] * len(state.players[1 - your_index].prize),
        opponent_hand=[1] * state.players[1 - your_index].handCount,
        opponent_active=[
            1072
        ] if len(active) > 0 and active[0] is None else []
    )
    
    node = Node(search_state)

    result = optimal_path(node, network)
    if result.max_option == None:
        return random_agent(obs_dict)

    return result.max_option

## Define Reward Function

### Subtask:
Implement a `reward_function` that takes the game observation and returns a reward based on the number of Pokémon each player defeated. For now, the reward will be a normalized difference in defeated Pokémon.

**Reasoning**:
This code block defines the `reward_function`. It calculates the reward based on the game's final observation, specifically the difference in Pokémon defeated by each player, and normalizes it by 6 (assuming a maximum of 6 Pokémon per player). This will serve as a basic signal for the agent to learn from.

In [20]:
def reward_function(final_obs_dict, current_player_index):
    """Return a plain Python float so rewards do not retain device tensors."""
    obs = to_observation_class(final_obs_dict)
    opponent_player_index = 1 if current_player_index == 0 else 0

    if len(obs.current.players[current_player_index].prize) < 6 and len(obs.current.players[opponent_player_index].prize) < 6:
        return 1.0 if obs.current.result == current_player_index else -1.0

    player_prizes = len(obs.current.players[current_player_index].prize)
    opponent_prizes = len(obs.current.players[opponent_player_index].prize)
    return float((player_prizes - opponent_prizes) / 6.0)

print("Reward function uses plain floats to keep episode memory small.")


Reward function uses plain floats to keep episode memory small.


## Implement Self-Play Game Simulation


In [21]:
from collections import deque
import time

# Hard memory ceiling for per-player experience collection.
MAX_EPISODE_EXPERIENCES = 256
PERFORMANCE_EPSILON_THRESHOLD = 0.10


def play_one_episode(tState0, tState1, max_turns=1000):
    """Play one game while retaining at most 256 states per player."""
    obs_dict_current, _ = battle_start(tState0.deck, tState1.deck)
    player0_experiences = deque(maxlen=MAX_EPISODE_EXPERIENCES)
    player1_experiences = deque(maxlen=MAX_EPISODE_EXPERIENCES)

    try:
        for turn in range(max_turns):
            obs = to_observation_class(obs_dict_current)
            current_player_index = obs.current.yourIndex
            tState = tState0 if current_player_index == 0 else tState1

            choice = dfs_agent(obs_dict_current, tState)
            if not choice:
                choice = random_agent(obs_dict_current)

            if len(choice) == 1 and obs.select.option[choice[0]].type == 14:
                # Only retain a bounded number of training states.
                if current_player_index == 0:
                    player0_experiences.append(encode_state(obs))
                else:
                    player1_experiences.append(encode_state(obs))

            next_obs = battle_select(choice)
            if next_obs is None:
                raise RuntimeError("battle_select returned None")
            obs_dict_current = next_obs

            if obs_dict_current["current"]["result"] >= 0:
                break
        else:
            print(f"WARNING: episode hit max_turns={max_turns}; treating as a draw")

        final_reward_p0 = reward_function(obs_dict_current, 0)
        final_reward_p1 = reward_function(obs_dict_current, 1)
        winner_index = int(obs_dict_current["current"].get("result", -1))

        # Only start tallying once the individual deck has reached low epsilon.
        tState0.record_result(0, winner_index, PERFORMANCE_EPSILON_THRESHOLD)
        if tState1 is not tState0:
            tState1.record_result(1, winner_index, PERFORMANCE_EPSILON_THRESHOLD)

        return list(player0_experiences), final_reward_p0, list(player1_experiences), final_reward_p1
    finally:
        tState0.update_epsilon()
        tState1.update_epsilon()
        battle_finish()
        # The next episode should not inherit any dead Python objects.
        del player0_experiences, player1_experiences

print("play_one_episode updated with a hard per-player RAM ceiling and low-epsilon tallying.")


play_one_episode updated with a hard per-player RAM ceiling and low-epsilon tallying.


Run generation

In [22]:
def run_generation(num_episodes, tState0, tState1=None, max_batch_size=256):
    """Compatibility wrapper: train immediately after each episode; never accumulate a generation."""
    if tState1 is None:
        tState1 = tState0

    losses = []
    for _ in range(num_episodes):
        p0, r0, p1, r1 = play_one_episode(tState0, tState1)
        loss0 = tState0.train(p0, [r0] * len(p0), max_batch_size=max_batch_size)
        loss1 = tState1.train(p1, [r1] * len(p1), max_batch_size=max_batch_size)
        if loss0 is not None:
            losses.append(loss0)
        if loss1 is not None:
            losses.append(loss1)
        del p0, p1
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return sum(losses) / len(losses) if losses else None

print("run_generation now trains episode-by-episode and does not build generation-sized RAM buffers.")


run_generation now trains episode-by-episode and does not build generation-sized RAM buffers.


### Search safety
The original exhaustive DFS had no bound and could appear stuck. The search above now has explicit depth, node, and branching limits, and the episode loop has a turn limit.

# Run Training Loop

In [23]:
import csv
import time
import gc

# ----- Training configuration -----
INPUT_DIM = 13557
TARGET_TRAINING_HOURS = 0.3
TARGET_TRAINING_SECONDS = TARGET_TRAINING_HOURS * 60 * 60
WARMUP_EPISODES = 10
EPISODES_PER_PROGRESS_REPORT = 25
PERFORMANCE_REPORT_EVERY = 50
PERFORMANCE_EPSILON_THRESHOLD = 0.10
MAX_BATCH_SIZE = 256
MAX_TURNS_PER_EPISODE = 1000

# One shared network + ONE optimizer avoids keeping a separate Adam state for every deck.
model = StateValueNetwork(INPUT_DIM).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
states = [
    TrainingState(
        model,
        deck,
        optimizer=optimizer,
        deck_name=predefined_deck_names[i],
        epsilon=1.0,
        epsilon_min=0.1,
        epsilon_decay=0.995,
    )
    for i, deck in enumerate(predefined_decks)
]

print("Device:", device)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
print(f"Loaded {len(states)} decks")
print(f"Experience cap per player/episode: {MAX_EPISODE_EXPERIENCES}")
print(f"Training target: {TARGET_TRAINING_HOURS:.1f} hours")

# ----- Short warm-up -----
for episode in range(WARMUP_EPISODES):
    state0 = random.choice(states)
    state1 = random.choice(states)
    start = time.perf_counter()
    p0, r0, p1, r1 = play_one_episode(state0, state1, max_turns=MAX_TURNS_PER_EPISODE)
    loss0 = state0.train(p0, [r0] * len(p0), max_batch_size=MAX_BATCH_SIZE)
    loss1 = state1.train(p1, [r1] * len(p1), max_batch_size=MAX_BATCH_SIZE)
    del p0, p1
    gc.collect()
    elapsed = time.perf_counter() - start
    print(f"warmup {episode + 1}/{WARMUP_EPISODES} - {elapsed:.2f}s - loss0={loss0} loss1={loss1}")

# ----- Main training: wall-clock controlled so it actually targets ~2 hours -----
training_start = time.perf_counter()
next_perf_report_games = PERFORMANCE_REPORT_EVERY
episode = 0
loss_window = []

while (time.perf_counter() - training_start) < TARGET_TRAINING_SECONDS:
    episode += 1
    state0 = random.choice(states)
    state1 = random.choice(states)

    p0, r0, p1, r1 = play_one_episode(
        state0,
        state1,
        max_turns=MAX_TURNS_PER_EPISODE,
    )

    loss0 = state0.train(p0, [r0] * len(p0), max_batch_size=MAX_BATCH_SIZE)
    loss1 = state1.train(p1, [r1] * len(p1), max_batch_size=MAX_BATCH_SIZE)
    if loss0 is not None:
        loss_window.append(loss0)
    if loss1 is not None:
        loss_window.append(loss1)

    del p0, p1
    if episode % 5 == 0:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    now = time.perf_counter()
    if episode % EPISODES_PER_PROGRESS_REPORT == 0:
        elapsed = now - training_start
        remaining = max(0.0, TARGET_TRAINING_SECONDS - elapsed)
        avg_loss = sum(loss_window[-100:]) / len(loss_window[-100:]) if loss_window else 0.0
        avg_eps = sum(s.epsilon for s in states) / len(states) if states else 0.0
        print(
            f"train episode={episode} | elapsed={elapsed/3600:.2f}h | "
            f"remaining~{remaining/3600:.2f}h | avg_epsilon={avg_eps:.3f} | "
            f"recent_loss={avg_loss:.5f}"
        )
        # Print deck tallies once exploration has become low.
        total_eval_games = sum(s.performance["games"] for s in states)
        if total_eval_games >= next_perf_report_games:
            print("\n=== LOW-EPSILON DECK PERFORMANCE ===")
            for idx, s in enumerate(states):
                p = s.performance
                if p["games"]:
                    print(
                        f"Deck {idx+1} ({s.deck_name}): "
                        f"{p['wins']}-{p['losses']}-{p['draws']} "
                        f"({s.win_rate()*100:.1f}% win rate), epsilon={s.epsilon:.3f}"
                    )
            print("===================================\n")
            next_perf_report_games += PERFORMANCE_REPORT_EVERY

print(f"Training finished after {episode} episodes and {(time.perf_counter()-training_start)/3600:.2f} hours.")

# ----- Final performance table -----
print("\n========== FINAL LOW-EPSILON DECK TALLY ==========")
for idx, s in enumerate(states):
    p = s.performance
    print(
        f"Deck {idx+1} ({s.deck_name}): "
        f"games={p['games']} wins={p['wins']} losses={p['losses']} draws={p['draws']} "
        f"win_rate={s.win_rate()*100:.1f}% epsilon={s.epsilon:.3f}"
    )
print("==================================================\n")

# Save one shared model and per-deck performance/deck metadata.
os.makedirs("trainingState", exist_ok=True)
model_path = "trainingState/shared_policy.pth"
torch.save(model.state_dict(), model_path)
for s in states:
    s.save_deck_metadata("trainingState")

with open("trainingState/deck_performance_summary.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["deck_index", "deck_label", "games", "wins", "losses", "draws", "win_rate", "epsilon"])
    for idx, s in enumerate(states, start=1):
        p = s.performance
        writer.writerow([
            idx,
            s.deck_name,
            p["games"],
            p["wins"],
            p["losses"],
            p["draws"],
            s.win_rate(),
            s.epsilon,
        ])

print(f"Saved shared model to {model_path}")
print("Saved per-deck performance tallies to trainingState/deck_performance_summary.csv")


Device: cuda
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8
Loaded 16 decks
Experience cap per player/episode: 256
Training target: 0.3 hours
warmup 1/10 - 1.43s - loss0=0.9081887006759644 loss1=1.2083858251571655
warmup 2/10 - 0.55s - loss0=0.005377542693167925 loss1=0.001350835314951837
warmup 3/10 - 1.36s - loss0=0.029546119272708893 loss1=0.03233763575553894
warmup 4/10 - 2.63s - loss0=1.0033838748931885 loss1=1.0886224508285522
warmup 5/10 - 0.45s - loss0=0.04025889188051224 loss1=0.019892271608114243
warmup 6/10 - 0.57s - loss0=0.46539226174354553 loss1=0.455247163772583
warmup 7/10 - 0.43s - loss0=0.0002370192378293723 loss1=8.175355469575152e-05
warmup 8/10 - 1.68s - loss0=0.09653402864933014 loss1=0.12350635975599289
warmup 9/10 - 2.74s - loss0=0.9845219850540161 loss1=1.0522384643554688
warmup 10/10 - 0.43s - loss0=0.03967740014195442 loss1=0.01941652037203312
train episode=25 | elapsed=0.02h | remaining~0.28h | avg_epsilon=0.978 | recent_loss=0.26058
train episode=50 